In [2]:
using ITensors
using ITensorMPS
using Random

using LinearAlgebra
using Statistics
#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end

function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end

function random_cdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i in 1:2:(L - Nup_extra)
        state[i] = "UpDn"
        Nup-=1
    end
    if Nup != 0
        state[L] = "Up"
    end
    return state
end 

random_cdw_state (generic function with 1 method)

In [ ]:
L = 7
sites = siteinds("Electron", L; conserve_qns=true)

maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

nsweeps = 10

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

state = random_metallic_state(L, Nup, Ndn)

println(state)

psi0 = random_mps(sites, state; linkdims=10)

U = 0.5 
V = -7.5
J = 1.0

H = H_EHM(L, J, U, V, sites)

energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff)

["Up", "Dn", "Up", "Dn", "Up", "UpDn", "Emp"]


In [ ]:
  N = 8
  m = 4

  s = siteinds("Electron", N; conserve_qns=true)
  psi = random_mps(s, n -> isodd(n) ? "Up" : "Dn"; linkdims=m)
  
  Cuu = correlation_matrix(psi, "Cdagup", "Cup")

- trying to create a non-uniform mesh grid to compute the phase diagram.

In [ ]:
# param1_segment1 = range(0.0, stop=0.9, length=10)
# param1_segment2 = range(0.91, stop=1.09, length=40)
# param1_segment3 = range(1.1, stop=2.0, length=10)
# param1_values = vcat(collect(param1_segment1), collect(param1_segment2), collect(param1_segment3))
# param2_values = collect(range(0.0, stop=1.0, length=50))

In [ ]:
N = 2
m = 4

s = siteinds("Electron", N; conserve_qns=true)

psi = productMPS(s, ["Up", "Dn"])

In [ ]:
rho_1 = build_1_particle_rdm(psi)

println(tr(rho_1))
println("is hermitian ? ", ishermitian(rho_1))

In [ ]:
real(rho_1)

In [ ]:
Cupup = correlation_matrix(psi, "Cdagup", "Cup")
Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")

println("Cupup = ", Cupup)
println("Cupdn = ", Cupdn)
println("Cdnup = ", Cdnup)
println("Cdndn = ", Cdndn)

In [ ]:
E_p = Ep(rho_1, N) - log2(N)
println("E_p = ", E_p)

In [ ]:
function density_operators(N, psi, sites)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
        orthogonalize!(psi, j)
        psidag_j = dag(prime(psi[j], "Site"))
        upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
        dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
        updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end

upd, dnd, updn = density_operators(N, psi, sites)

In [ ]:
magnetization = (upd .- dnd) / 2
print(magnetization)

In [ ]:
charge_density = (upd .+ dnd)
print(charge_density)

In [ ]:
function m_sdw(L, Sj)
    m_sdw = 0.0 
    for j in 1:L 
        println(j, " ", Sj[j])
        m_sdw += (-1)^(j) * Sj[j]
    end
    return m_sdw / L
end
function m_cdw(L, nj)
    m_cdw = 0.0
    for j in 1:L 
        println(j, " ", nj[j])
        m_cdw += (-1)^(j) * (nj[j] - 1)
    end
    return m_cdw / L 
end 

In [ ]:
println("|m_sdw| = ", abs(m_sdw(N, magnetization)))
println("|m_cdw| = ", abs(m_cdw(N, charge_density)))

2-rdm

In [ ]:
using ITensors, ITensorMPS

N = 10
sites = siteinds("Electron", N)

os = OpSum()

os += "Cdagup", 2, "Cdagup", 4, "Cdn", 3, "Cdn", 1

# Again, you can add more terms and coefficients as needed.

my_operator_MPO = MPO(os, sites)

println("\nCreated MPO for the spinful operator:")
println(os)

In [ ]:
spins = ["up", "dn"]

In [ ]:
function random_ps_state(L, Nup, Ndn)
    state = fill("Emp", L)
    
    Ndbl = min(Nup, Ndn)
    Nup -= Ndbl
    Ndn -= Ndbl
    
    for i in 1:Ndbl
        state[i] = "UpDn"
    end
    
    next_site = Ndbl + 1
    
    if Nup > 0
        state[next_site] = "Up"
        next_site += 1
    elseif Ndn > 0
        state[next_site] = "Dn"
        next_site += 1
    end
    return state
end



L = 7
sites = siteinds("Electron", L; conserve_qns=true)

maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

nsweeps = 10



In [4]:
N = 5
sites = siteinds("Electron", N; conserve_qns=true)

Npart = floor(Int, N/2) 
Nup = Npart + N % 2 
Ndn = N - Nup 

state = random_metallic_state(N, Nup, Ndn)

println(state)

psi = random_mps(sites, state; linkdims=4)

i_site = 1
j_site = 3
l_site = 2
k_site = 4

ampo_op = OpSum()
ampo_op += "Cdagup", i_site, "Cdagup", j_site, "Cup", l_site, "Cdn", k_site

O_mpo = MPO(ampo_op, sites)

exp_val = inner(prime(psi), O_mpo, psi)

println("Expectation value of operator: ", exp_val)

["Up", "Dn", "Up", "UpDn", "Emp"]
Expectation value of operator: 0.0


In [29]:
function spin_correlators_mpo(state, sites, i, j, l, k)
    spins = ["up", "dn"]

    matrix_block = zeros(ComplexF64, 4, 4)

    for i_spin in 1:2
        for j_spin in 1:2
            
            row = 2 * (i_spin - 1) + j_spin

            for l_spin in 1:2
                for k_spin in 1:2

                    col = 2 * (l_spin - 1) + k_spin

                    ampo_op = OpSum()

                    ampo_op += "Cdag" * spins[i_spin], i, "Cdag" * spins[j_spin], j,
                                "C" * spins[l_spin], l, "C" * spins[k_spin], k

                    O_mpo = MPO(ampo_op, sites)
                    
                    matrix_block[row, col] = inner(prime(state), O_mpo, state)
                    
                end
            end
        end
    end
    return matrix_block
end


function build_2_rdm(state, L)

    L = length(state)
    rho_2 = zeros(ComplexF64, (2*L)^2, (2*L)^2) 
    
    # (L^2) x (L^2) blocks of spin correlators.

    for i in 1:L 
        for j in 1:L 
            for l in 1:L 
                for k in 1:L
                    
                    block_spin_correlators = spin_correlators_mpo(state, sites, i, j, l, k)

                    # Compute block position
                    row_block = 2*(i-1)*L + 2*(j-1)
                    col_block = 2*(l-1)*L + 2*(k-1)

                    # Fill in the 4x4 block
                    rho_2[row_block+1:row_block+4, col_block+1:col_block+4] = block_spin_correlators
                end
            end
        end
    end
    return rho_2
end

spin_correlators_mpo(psi, sites, 1, 2, 3, 4)

rho_2 = build_2_rdm(psi, sites)

100×100 Matrix{ComplexF64}:
 0.0+0.0im         0.0+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im     -0.2134+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im   -0.104186+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im   0.0166169+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  -0.0141298+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.00408451+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
    ⋮                         ⋱                        
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im         0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0

In [34]:
using LinearAlgebra
println(tr(rho_2))

-12.459628016252374 + 0.0im
